In [8]:
!pip install -q accelerate==0.21.0 peft==0.4.0 bitsandbytes==0.40.2 transformers==4.31.0 trl==0.4.7 tokenizers==0.13.3 sentencepiece tensorboard

In [1]:
import os
import torch
from datasets import load_dataset
from datasets.arrow_dataset import Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    HfArgumentParser,
    TrainingArguments,
    pipeline,
    logging,
)
from peft import LoraConfig, PeftModel
from trl import SFTTrainer

import pandas as pd
import pyarrow as pa

In [2]:
size = "13"

# The model that you want to train from the Hugging Face hub
model_name = f"NousResearch/llama-2-{size}b-chat-hf"

# Fine-tuned model name
new_model = f"llama-2-{size}b-generate-questions"

################################################################################
# QLoRA parameters
################################################################################

# LoRA attention dimension
lora_r = 64

# Alpha parameter for LoRA scaling
lora_alpha = 16

# Dropout probability for LoRA layers
lora_dropout = 0.1

################################################################################
# bitsandbytes parameters
################################################################################

# Activate 4-bit precision base model loading
use_4bit = True

# Compute dtype for 4-bit base models
bnb_4bit_compute_dtype = "float16"

# Quantization type (fp4 or nf4)
bnb_4bit_quant_type = "nf4"

# Activate nested quantization for 4-bit base models (double quantization)
use_nested_quant = False

################################################################################
# TrainingArguments parameters
################################################################################

# Output directory where the model predictions and checkpoints will be stored
output_dir = "./results"

# Number of training epochs
num_train_epochs = 25

# Enable fp16/bf16 training (set bf16 to True with an A100)
fp16 = False
bf16 = False

# Batch size per GPU for training
per_device_train_batch_size = 4

# Batch size per GPU for evaluation
per_device_eval_batch_size = 4

# Number of update steps to accumulate the gradients for
gradient_accumulation_steps = 1

# Enable gradient checkpointing
gradient_checkpointing = True

# Maximum gradient normal (gradient clipping)
max_grad_norm = 0.3

# Initial learning rate (AdamW optimizer)
learning_rate = 2e-4

# Weight decay to apply to all layers except bias/LayerNorm weights
weight_decay = 0.001

# Optimizer to use
optim = "paged_adamw_32bit"

# Learning rate schedule (constant a bit better than cosine)
lr_scheduler_type = "constant"

# Number of training steps (overrides num_train_epochs)
max_steps = -1

# Ratio of steps for a linear warmup (from 0 to learning rate)
warmup_ratio = 0.03

# Group sequences into batches with same length
# Saves memory and speeds up training considerably
group_by_length = True

# Save checkpoint every X updates steps
save_steps = 25

# Log every X updates steps
logging_steps = 25

################################################################################
# SFT parameters
################################################################################

# Maximum sequence length to use
max_seq_length = None

# Pack multiple short examples in the same input sequence to increase efficiency
packing = False

# Load the entire model on the GPU 0
device_map = {"": 0}

In [3]:
data = pd.read_csv("dataset_prompts/dataset.csv")
data

,file,prompt,response,text
0,example5.odt,Gere uma questão de matematica em formato JSON...,"[{""text"": ""Teresa tem 7 pirulitos e deu 3 para...",<s>[INST] Gere uma questão de matematica em fo...
1,example2.odt,Gere uma questão de matematica em formato JSON...,"[{""text"": ""Faça a dos conjuntos e ligue com o...",<s>[INST] Gere uma questão de matematica em fo...
2,example4.odt,Gere uma questão de matematica em formato JSON...,"[{""text"": ""Observe as figuras e responda as se...",<s>[INST] Gere uma questão de matematica em fo...
3,example1.odt,Gere uma questão de matematica em formato JSON...,"[{""text"": ""Faça a dos conjuntos e escreva o r...",<s>[INST] Gere uma questão de matematica em fo...
4,example3.odt,Gere uma questão de matematica em formato JSON...,"[{""text"": ""Escreva os números que faltam para ...",<s>[INST] Gere uma questão de matematica em fo...
5,example6.odt,Gere uma questão de matematica em formato JSON...,"[{""text"": ""Márcio tem 5 balões de são joão, se...",<s>[INST] Gere uma questão de matematica em fo...
6,example8.odt,Gere uma questão de matematica em formato JSON...,"[{""text"": ""Maria tem 5 pirulitos, e ganhou mai...",<s>[INST] Gere uma questão de matematica em fo...
7,example5.odt,Gere uma questão de matematica em formato JSON...,"[{""text"": ""Qual a soma dos números pares abaix...",<s>[INST] Gere uma questão de matematica em fo...
8,example7.odt,Gere uma questão de matematica em formato JSON...,"[{""text"": ""Qual a soma dos números abaixo?""},...",<s>[INST] Gere uma questão de matematica em fo...
9,example2.odt,Gere uma questão de matematica em formato JSON...,"[{""text"": ""Faça a soma dos conjuntos e ligue c...",<s>[INST] Gere uma questão de matematica em fo...


In [4]:
print(data.iloc[0]['text'])

<s>[INST] Gere uma questão de matematica em formato JSON que ajude crianças do 
ensino fundamental os principios da subtração de maneira lúdica e criativa.
Utilize uma ou mais das seguintes imagens:

numero_6_preto_branco.png :: figura do número 6 em preto e branco
numero_9_preto_branco.png :: figura do número 9 em preto e branco
boneca.png :: imagem de uma boneca preto e branco
numero_8_preto_branco.png :: figura do número 8 em preto e branco
numero_5_preto_branco.png :: figura do número 5 em preto e branco
abacaxi_preto_branco.png :: figura de um abacaxi em preto e branco
numero_4_preto_branco.png :: figura do número 4 em preto e branco
sinal_mais.png :: imagem de um sinal de mais
dado_5.png :: imagem de um dado mostrando o lado com número 5
carrinho.png :: imagem de um carro de brinquedo em preto e branco
dado_2.png :: imagem de um dado mostrando o lado com número 2
numero_1_preto_branco.png :: figura do número 1 em preto e branco
numero_2_preto_branco.png :: figura do número 2 em p

In [5]:
dataset_name = "questions"
dataset = Dataset(pa.Table.from_pandas(data[['text']]))
dataset

Dataset({
    features: ['text'],
    num_rows: 18
})

In [6]:
# Load tokenizer and model with QLoRA configuration
compute_dtype = getattr(torch, bnb_4bit_compute_dtype)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=use_4bit,
    bnb_4bit_quant_type=bnb_4bit_quant_type,
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=use_nested_quant,
)

# Check GPU compatibility with bfloat16
if compute_dtype == torch.float16 and use_4bit:
    major, _ = torch.cuda.get_device_capability()
    if major >= 8:
        print("=" * 80)
        print("Your GPU supports bfloat16: accelerate training with bf16=True")
        print("=" * 80)

# Load base model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map=device_map
)
model.config.use_cache = False
model.config.pretraining_tp = 1

# Load LLaMA tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True, use_fast=False)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right" # Fix weird overflow issue with fp16 training

# Load LoRA configuration
peft_config = LoraConfig(
    lora_alpha=lora_alpha,
    lora_dropout=lora_dropout,
    r=lora_r,
    bias="none",
    task_type="CAUSAL_LM",
)

# Set training parameters
training_arguments = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=num_train_epochs,
    per_device_train_batch_size=per_device_train_batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    optim=optim,
    save_steps=save_steps,
    logging_steps=logging_steps,
    learning_rate=learning_rate,
    weight_decay=weight_decay,
    fp16=fp16,
    bf16=bf16,
    max_grad_norm=max_grad_norm,
    max_steps=max_steps,
    warmup_ratio=warmup_ratio,
    group_by_length=group_by_length,
    lr_scheduler_type=lr_scheduler_type,
    report_to="tensorboard"
)

# Set supervised fine-tuning parameters
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    tokenizer=tokenizer,
    args=training_arguments,
    packing=packing,
)

# Train model
trainer.train()

# Save trained model
trainer.model.save_pretrained(new_model)

Your GPU supports bfloat16: accelerate training with bf16=True


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

/home/matheus.lisboa/anaconda3/lib/python3.11/site-packages/peft/utils/other.py:102: FutureWarning: prepare_model_for_int8_training is deprecated and will be removed in a future version. Use prepare_model_for_kbit_training instead.
  warnings.warn(
/home/matheus.lisboa/anaconda3/lib/python3.11/site-packages/trl/trainer/sft_trainer.py:159: UserWarning: You didn't pass a `max_seq_length` argument to the SFTTrainer, this will default to 1024
  warnings.warn(


Map:   0%|          | 0/18 [00:00<?, ? examples/s]

/home/matheus.lisboa/anaconda3/lib/python3.11/site-packages/torch/utils/checkpoint.py:429: UserWarning: torch.utils.checkpoint: please pass in use_reentrant=True or use_reentrant=False explicitly. The default value of use_reentrant will be updated to be False in the future. To maintain current behavior, pass use_reentrant=True. It is recommended that you use use_reentrant=False. Refer to docs for more details on the differences between the two variants.
  warnings.warn(


Step,Training Loss
25,0.595400
50,0.153500
75,0.070900
100,0.031100
125,0.019000


/home/matheus.lisboa/anaconda3/lib/python3.11/site-packages/torch/utils/checkpoint.py:429: UserWarning: torch.utils.checkpoint: please pass in use_reentrant=True or use_reentrant=False explicitly. The default value of use_reentrant will be updated to be False in the future. To maintain current behavior, pass use_reentrant=True. It is recommended that you use use_reentrant=False. Refer to docs for more details on the differences between the two variants.
  warnings.warn(
/home/matheus.lisboa/anaconda3/lib/python3.11/site-packages/torch/utils/checkpoint.py:429: UserWarning: torch.utils.checkpoint: please pass in use_reentrant=True or use_reentrant=False explicitly. The default value of use_reentrant will be updated to be False in the future. To maintain current behavior, pass use_reentrant=True. It is recommended that you use use_reentrant=False. Refer to docs for more details on the differences between the two variants.
  warnings.warn(
/home/matheus.lisboa/anaconda3/lib/python3.11/site

In [10]:
# Ignore warnings
logging.set_verbosity(logging.CRITICAL)

# Run text generation pipeline with our next model
prompt = """
Gere uma questão em formato JSON que ajude crianças do ensino fundamental
os principios da adição de maneira lúdica e criativa.
Utilize uma ou mais das seguintes imagens:

boneca.png :: imagem de uma boneca preto e branco
carrinho.png :: imagem de um carro de brinquedo em preto e branco
"""
pipe = pipeline(task="text-generation", model=model, tokenizer=tokenizer, max_length=1000)
result = pipe(f"<s>[INST] {prompt} [/INST]")
print(result[0]['generated_text'])

<s>[INST] 
Gere uma questão em formato JSON que ajude crianças do ensino fundamental
os principios da adição de maneira lúdica e criativa.
Utilize uma ou mais das seguintes imagens:

boneca.png :: imagem de uma boneca preto e branco
carrinho.png :: imagem de um carro de brinquedo em preto e branco
 [/INST] [{"text": "João tem 5 bonecas e recebe mais de 7 bonecas, faça a adição:"}, {"image": "boneca.png", "width": "2.332cm", "height": "2.456cm", "x": "4.337cm", "y": "5.423cm"}, {"image": "boneca.png", "width": "2.265cm", "height": "2.397cm", "x": "8.975cm", "y": "6.556cm"}, {"image": "boneca.png", "width": "2.227cm", "height": "2.478cm", "x": "11.09cm", "y": "4.246cm"}, {"image": "boneca.png", "width": "2.144cm", "height": "2.286cm", "x": "11.65cm", "y": "7.097cm"}, {"image": "boneca.png", "width": "2.059cm", "height": "2.227cm", "x": "5.72cm", "y": "7.803cm"}]  [5] :: figura de uma boneca com wings que ajuda a representar principios matemáticos.
 [11] :: figura de uma boneca com wings 

In [ ]:
"""
Gere uma questão em formato JSON que ajude crianças do ensino fundamental
os principios da adição de maneira lúdica e criativa.
Utilize uma ou mais das seguintes imagens:

numero_6_preto_branco.png :: figura do número 6 em preto e branco
numero_9_preto_branco.png :: figura do número 9 em preto e branco
boneca.png :: imagem de uma boneca preto e branco
numero_8_preto_branco.png :: figura do número 8 em preto e branco
numero_5_preto_branco.png :: figura do número 5 em preto e branco
abacaxi_preto_branco.png :: figura de um abacaxi em preto e branco
numero_4_preto_branco.png :: figura do número 4 em preto e branco
sinal_mais.png :: imagem de um sinal de mais
dado_5.png :: imagem de um dado mostrando o lado com número 5
carrinho.png :: imagem de um carro de brinquedo em preto e branco
dado_2.png :: imagem de um dado mostrando o lado com número 2
numero_1_preto_branco.png :: figura do número 1 em preto e branco
numero_2_preto_branco.png :: figura do número 2 em preto e branco
numero_7_preto_branco.png :: figura do número 7 em preto e branco
sinal_igual.png :: imagem de um sinal de igual
maca_preto_branco.png :: figura de uma maçã em preto e branco
dado_4.png :: imagem de um dado mostrando o lado com número 4
dado_1.png :: imagem de um dado mostrando o lado com número 1
dado_3.png :: imagem de um dado mostrando o lado com número 3
dado_6.png :: imagem de um dado mostrando o lado com número 6
numero_3_preto_branco.png :: figura do número 3 em preto e branco
"""

In [ ]:
# base_model = AutoModelForCausalLM.from_pretrained(
#     model_name,
#     low_cpu_mem_usage=True,
#     return_dict=True,
#     torch_dtype=torch.float16,
#     device_map=device_map,
# )
# model = PeftModel.from_pretrained(base_model, new_model)
# model = model.merge_and_unload()

# # Reload tokenizer to save it
# tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
# tokenizer.pad_token = tokenizer.eos_token
# tokenizer.padding_side = "right"